# SQL Sales Analysis Dashboard

Comprehensive SQL-based sales analysis using SQLite, Pandas, and Visualization libraries. This notebook demonstrates how to load sales data, execute SQL queries, and visualize insights.

In [ ]:
import sys
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime
import warnings

warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

# Set visualization style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 10

print("✓ All libraries imported successfully!")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

In [ ]:
# Database connection
db_path = "sales_analysis.db"
csv_path = Path("sales_dashboard_project/data/sales_data.csv")

# Create connection
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# Load CSV data into database if not already loaded
try:
    cursor.execute("SELECT COUNT(*) FROM sales")
    count = cursor.fetchone()[0]
    print(f"✓ Database already has {count} records")
except sqlite3.OperationalError:
    # Table doesn't exist, load from CSV
    print(f"Loading data from {csv_path}")
    df_raw = pd.read_csv(csv_path)
    df_raw.to_sql('sales', conn, if_exists='replace', index=False)
    conn.commit()
    print(f"✓ Loaded {len(df_raw)} records into database")

print(f"✓ Connected to database: {db_path}")

In [ ]:
# Get database schema
query_schema = "PRAGMA table_info(sales)"
df_schema = pd.read_sql_query(query_schema, conn)
print("📊 Database Schema:")
print(df_schema.to_string(index=False))

# Get data overview
query_overview = """
SELECT 
    COUNT(*) as Total_Records,
    COUNT(DISTINCT Sales_Rep) as Total_Sales_Reps,
    COUNT(DISTINCT Region) as Total_Regions,
    COUNT(DISTINCT Product_Category) as Total_Categories,
    COUNT(DISTINCT Sales_Channel) as Total_Channels
FROM sales
"""
df_overview = pd.read_sql_query(query_overview, conn)
print("\n📈 Data Overview:")
print(df_overview.to_string(index=False))

# Sample of data
print("\n📋 Sample Data:")
df_sample = pd.read_sql_query("SELECT * FROM sales LIMIT 5", conn)
print(df_sample.to_string(index=False))

In [ ]:
query_product = """
SELECT 
    Product_Category,
    COUNT(*) as Sales_Count,
    SUM(Sales_Amount) as Total_Revenue,
    AVG(Sales_Amount) as Avg_Sale_Value,
    SUM(Quantity_Sold) as Total_Units_Sold,
    ROUND(SUM(Sales_Amount) * 100.0 / (SELECT SUM(Sales_Amount) FROM sales), 2) as Revenue_Percentage
FROM sales
GROUP BY Product_Category
ORDER BY Total_Revenue DESC
"""

df_product = pd.read_sql_query(query_product, conn)
print("🏷️ Sales by Product Category:")
print(df_product.to_string(index=False))

# Top products
query_top_products = """
SELECT 
    Product_ID,
    Product_Category,
    COUNT(*) as Sales_Count,
    ROUND(SUM(Sales_Amount), 2) as Total_Revenue,
    SUM(Quantity_Sold) as Units_Sold
FROM sales
GROUP BY Product_ID, Product_Category
ORDER BY Total_Revenue DESC
LIMIT 10
"""

df_top_products = pd.read_sql_query(query_top_products, conn)
print("\n🎯 Top 10 Products by Revenue:")
print(df_top_products.to_string(index=False))

In [ ]:
# Monthly revenue trend
query_monthly = """
SELECT 
    strftime('%Y-%m', Sale_Date) as Month,
    COUNT(*) as Transaction_Count,
    ROUND(SUM(Sales_Amount), 2) as Monthly_Revenue,
    ROUND(AVG(Sales_Amount), 2) as Avg_Transaction_Value,
    SUM(Quantity_Sold) as Total_Units
FROM sales
GROUP BY strftime('%Y-%m', Sale_Date)
ORDER BY Month
"""

df_monthly = pd.read_sql_query(query_monthly, conn)
print("📅 Monthly Sales Trend:")
print(df_monthly.to_string(index=False))

# Seasonal analysis (by quarter)
query_seasonal = """
SELECT 
    strftime('%Y', Sale_Date) as Year,
    'Q' || CAST(((strftime('%m', Sale_Date) - 1) / 3) + 1 AS INTEGER) as Quarter,
    COUNT(*) as Transaction_Count,
    ROUND(SUM(Sales_Amount), 2) as Quarterly_Revenue,
    ROUND(AVG(Sales_Amount), 2) as Avg_Transaction
FROM sales
GROUP BY Year, Quarter
ORDER BY Year, Quarter
"""

df_seasonal = pd.read_sql_query(query_seasonal, conn)
print("\n🌍 Seasonal Analysis (Quarterly):")
print(df_seasonal.to_string(index=False))

# Day of week analysis
query_dow = """
SELECT 
    CASE CAST(strftime('%w', Sale_Date) AS INTEGER)
        WHEN 0 THEN 'Sunday'
        WHEN 1 THEN 'Monday'
        WHEN 2 THEN 'Tuesday'
        WHEN 3 THEN 'Wednesday'
        WHEN 4 THEN 'Thursday'
        WHEN 5 THEN 'Friday'
        WHEN 6 THEN 'Saturday'
    END as Day_of_Week,
    COUNT(*) as Transaction_Count,
    ROUND(SUM(Sales_Amount), 2) as Total_Sales,
    ROUND(AVG(Sales_Amount), 2) as Avg_Sale
FROM sales
GROUP BY strftime('%w', Sale_Date)
ORDER BY CAST(strftime('%w', Sale_Date) AS INTEGER)
"""

df_dow = pd.read_sql_query(query_dow, conn)
print("\n📆 Day of Week Analysis:")
print(df_dow.to_string(index=False))

In [ ]:
# Revenue by region
query_region = """
SELECT 
    Region,
    COUNT(*) as Transaction_Count,
    ROUND(SUM(Sales_Amount), 2) as Total_Revenue,
    ROUND(AVG(Sales_Amount), 2) as Avg_Sales,
    SUM(Quantity_Sold) as Total_Units
FROM sales
GROUP BY Region
ORDER BY Total_Revenue DESC
"""

df_region = pd.read_sql_query(query_region, conn)
print("📍 Revenue by Region:")
print(df_region.to_string(index=False))

# Sales rep performance
query_rep = """
SELECT 
    Sales_Rep,
    Region,
    COUNT(*) as Total_Transactions,
    ROUND(SUM(Sales_Amount), 2) as Total_Sales,
    ROUND(AVG(Sales_Amount), 2) as Avg_Transaction,
    SUM(Quantity_Sold) as Total_Items_Sold,
    ROUND(AVG(Discount) * 100, 2) as Avg_Discount_Percent
FROM sales
GROUP BY Sales_Rep, Region
ORDER BY Total_Sales DESC
"""

df_rep = pd.read_sql_query(query_rep, conn)
print("\n👥 Sales Rep Performance:")
print(df_rep.to_string(index=False))

In [ ]:
# Customer type analysis
query_customer = """
SELECT 
    Customer_Type,
    COUNT(*) as Customer_Count,
    ROUND(SUM(Sales_Amount), 2) as Total_Revenue,
    ROUND(AVG(Sales_Amount), 2) as Avg_Purchase_Value,
    ROUND(AVG(Discount) * 100, 2) as Avg_Discount_Percent
FROM sales
GROUP BY Customer_Type
ORDER BY Total_Revenue DESC
"""

df_customer = pd.read_sql_query(query_customer, conn)
print("👤 Customer Type Analysis:")
print(df_customer.to_string(index=False))

# Sales channel analysis
query_channel = """
SELECT 
    Sales_Channel,
    COUNT(*) as Transaction_Count,
    ROUND(SUM(Sales_Amount), 2) as Total_Revenue,
    ROUND(AVG(Sales_Amount), 2) as Avg_Transaction,
    SUM(Quantity_Sold) as Total_Units,
    ROUND(SUM(Sales_Amount) * 100.0 / (SELECT SUM(Sales_Amount) FROM sales), 2) as Revenue_Share
FROM sales
GROUP BY Sales_Channel
ORDER BY Total_Revenue DESC
"""

df_channel = pd.read_sql_query(query_channel, conn)
print("\n🔗 Sales Channel Analysis (Online vs Retail):")
print(df_channel.to_string(index=False))

# Payment method
query_payment = """
SELECT 
    Payment_Method,
    COUNT(*) as Transaction_Count,
    ROUND(SUM(Sales_Amount), 2) as Total_Revenue,
    ROUND(AVG(Sales_Amount), 2) as Avg_Amount
FROM sales
GROUP BY Payment_Method
ORDER BY Total_Revenue DESC
"""

df_payment = pd.read_sql_query(query_payment, conn)
print("\n💳 Payment Method Distribution:")
print(df_payment.to_string(index=False))

In [ ]:
# Overall KPIs
query_kpi = """
SELECT 
    COUNT(*) as Total_Transactions,
    ROUND(SUM(Sales_Amount), 2) as Total_Revenue,
    ROUND(AVG(Sales_Amount), 2) as Avg_Transaction_Value,
    ROUND(MIN(Sales_Amount), 2) as Min_Transaction,
    ROUND(MAX(Sales_Amount), 2) as Max_Transaction,
    SUM(Quantity_Sold) as Total_Units_Sold,
    COUNT(DISTINCT Sales_Rep) as Total_Sales_Reps,
    COUNT(DISTINCT Region) as Total_Regions
FROM sales
"""

df_kpi = pd.read_sql_query(query_kpi, conn)
print("📊 Overall Business KPIs:")
print(df_kpi.to_string(index=False))

# Profitability analysis
query_profit = """
SELECT 
    Product_Category,
    COUNT(*) as Sales_Count,
    ROUND(AVG(Unit_Cost), 2) as Avg_Cost,
    ROUND(AVG(Unit_Price), 2) as Avg_Price,
    ROUND(AVG(Unit_Price - Unit_Cost), 2) as Avg_Profit_Per_Unit,
    ROUND(AVG((Unit_Price - Unit_Cost) / Unit_Price * 100), 2) as Profit_Margin_Percent,
    ROUND(SUM(Quantity_Sold * (Unit_Price - Unit_Cost)), 2) as Total_Profit
FROM sales
GROUP BY Product_Category
ORDER BY Total_Profit DESC
"""

df_profit = pd.read_sql_query(query_profit, conn)
print("\n💰 Profitability Analysis by Category:")
print(df_profit.to_string(index=False))

In [ ]:
# Create visualizations
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Revenue by Region (Bar Chart)
df_region_sorted = df_region.sort_values('Total_Revenue', ascending=True)
axes[0, 0].barh(df_region_sorted['Region'], df_region_sorted['Total_Revenue'], color='steelblue')
axes[0, 0].set_xlabel('Total Revenue ($)', fontsize=11)
axes[0, 0].set_title('Revenue by Region', fontsize=13, fontweight='bold')
axes[0, 0].grid(axis='x', alpha=0.3)

# 2. Product Category Revenue (Pie Chart)
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A']
axes[0, 1].pie(df_product['Total_Revenue'], labels=df_product['Product_Category'], 
               autopct='%1.1f%%', colors=colors, startangle=90)
axes[0, 1].set_title('Revenue Distribution by Category', fontsize=13, fontweight='bold')

# 3. Monthly Revenue Trend (Line Chart)
df_monthly['Month'] = pd.to_datetime(df_monthly['Month'])
axes[1, 0].plot(df_monthly['Month'], df_monthly['Monthly_Revenue'], marker='o', 
                linewidth=2, markersize=6, color='green')
axes[1, 0].set_xlabel('Month', fontsize=11)
axes[1, 0].set_ylabel('Revenue ($)', fontsize=11)
axes[1, 0].set_title('Monthly Revenue Trend', fontsize=13, fontweight='bold')
axes[1, 0].tick_params(axis='x', rotation=45)
axes[1, 0].grid(True, alpha=0.3)

# 4. Sales Rep Performance (Top 5)
df_rep_top = df_rep.nlargest(5, 'Total_Sales')
axes[1, 1].barh(df_rep_top['Sales_Rep'], df_rep_top['Total_Sales'], color='coral')
axes[1, 1].set_xlabel('Total Sales ($)', fontsize=11)
axes[1, 1].set_title('Top 5 Sales Representatives', fontsize=13, fontweight='bold')
axes[1, 1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ Visualizations created successfully!")

In [ ]:
# Additional visualizations
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# 1. Customer Type Distribution
df_customer_sorted = df_customer.sort_values('Total_Revenue', ascending=True)
axes[0, 0].barh(df_customer_sorted['Customer_Type'], df_customer_sorted['Total_Revenue'], 
                color=['#3498db', '#e74c3c'])
axes[0, 0].set_xlabel('Total Revenue ($)', fontsize=11)
axes[0, 0].set_title('Revenue by Customer Type', fontsize=13, fontweight='bold')
axes[0, 0].grid(axis='x', alpha=0.3)

# 2. Sales Channel Comparison
df_channel_sorted = df_channel.sort_values('Total_Revenue', ascending=True)
axes[0, 1].barh(df_channel_sorted['Sales_Channel'], df_channel_sorted['Total_Revenue'], 
                color=['#2ecc71', '#f39c12'])
axes[0, 1].set_xlabel('Total Revenue ($)', fontsize=11)
axes[0, 1].set_title('Revenue by Sales Channel', fontsize=13, fontweight='bold')
axes[0, 1].grid(axis='x', alpha=0.3)

# 3. Day of Week Sales Pattern
df_dow_ordered = df_dow.copy()
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
df_dow_ordered['Day_of_Week'] = pd.Categorical(df_dow_ordered['Day_of_Week'], categories=day_order, ordered=True)
df_dow_ordered = df_dow_ordered.sort_values('Day_of_Week')
axes[1, 0].bar(range(len(df_dow_ordered)), df_dow_ordered['Total_Sales'], color='mediumpurple')
axes[1, 0].set_xticks(range(len(df_dow_ordered)))
axes[1, 0].set_xticklabels(df_dow_ordered['Day_of_Week'], rotation=45)
axes[1, 0].set_ylabel('Total Sales ($)', fontsize=11)
axes[1, 0].set_title('Sales Pattern by Day of Week', fontsize=13, fontweight='bold')
axes[1, 0].grid(axis='y', alpha=0.3)

# 4. Profit Margin by Category
df_profit_sorted = df_profit.sort_values('Profit_Margin_Percent', ascending=True)
axes[1, 1].barh(df_profit_sorted['Product_Category'], df_profit_sorted['Profit_Margin_Percent'], 
                color='lightcoral')
axes[1, 1].set_xlabel('Profit Margin (%)', fontsize=11)
axes[1, 1].set_title('Profit Margin by Product Category', fontsize=13, fontweight='bold')
axes[1, 1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ Additional visualizations created!")

In [ ]:
# Extract key insights
total_revenue = df_kpi['Total_Revenue'].values[0]
total_transactions = df_kpi['Total_Transactions'].values[0]
avg_transaction = df_kpi['Avg_Transaction_Value'].values[0]
top_product = df_product.iloc[0]
top_rep = df_rep.iloc[0]
top_region = df_region.iloc[0]

print("\n" + "="*80)
print("KEY INSIGHTS & RECOMMENDATIONS")
print("="*80 + "\n")

print(f"💰 Total Revenue: ${total_revenue:,.2f}")
print(f"📊 Total Transactions: {int(total_transactions):,}")
print(f"📈 Average Transaction Value: ${avg_transaction:,.2f}\n")

print(f"🏆 Top Performing Region: {top_region['Region']} (${top_region['Total_Revenue']:,.2f})")
print(f"👥 Top Sales Rep: {top_rep['Sales_Rep']} in {top_rep['Region']} (${top_rep['Total_Sales']:,.2f})")
print(f"🏷️ Top Product Category: {top_product['Product_Category']} ({top_product['Revenue_Percentage']}% of revenue)\n")

print("📊 Customer Distribution:")
print(f"   • New Customers: {df_customer[df_customer['Customer_Type'] == 'New']['Total_Revenue'].values[0]:,.2f}")
print(f"   • Returning Customers: {df_customer[df_customer['Customer_Type'] == 'Returning']['Total_Revenue'].values[0]:,.2f}\n")

print("🔗 Channel Performance:")
for idx, row in df_channel.iterrows():
    print(f"   • {row['Sales_Channel']}: ${row['Total_Revenue']:,.2f} ({row['Revenue_Share']}%)")

print("\n" + "="*80)

# Close database connection
conn.close()
print("✓ Database connection closed\n")
print("✅ SQL Sales Analysis Complete!")

## 11. Summary and Key Insights

Close the database connection and provide summary insights.

## 10. Additional Analysis Visualizations

Create additional charts for deeper insights.

## 9. Data Visualization

Create visual representations of sales data for better insights.

## 8. Key Performance Metrics

Calculate important KPIs and profitability metrics.

## 7. Customer Type and Channel Analysis

Analyze sales patterns by customer type and sales channel.

## 6. Regional and Sales Rep Performance

Analyze performance by region and individual sales representatives.

## 5. Sales Trends Over Time

Analyze monthly trends and seasonal patterns in sales data.

## 4. Sales Analysis by Product Category

Analyze revenue, quantity, and performance by product category.

## 3. Sales Data Overview

Examine the structure and summary statistics of sales data.

## 2. Connect to Database and Load Data

Establish SQLite connection and load sales data from CSV.

## 1. Import Required Libraries

Import necessary libraries for database connectivity, data manipulation, and visualization.